# Coffee Shop Sales Analysis

## Milestone 2: Sales analysis

Objective: Revenue calculations, store comparisons, product analysis, and sales trends.

In [2]:
import pandas as pd

from pathlib import Path

# Find the workbook from the project root or any folder within the project.
project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "data" / "Coffee Shop Sales.xlsx").is_file()),
    None
)
if project_root is None:
    raise FileNotFoundError(
        "Open the notebook from the Coffee Shop Analysis project folder "
        "or a folder inside it. Expected data/Coffee Shop Sales.xlsx."
    )
df = pd.read_excel(project_root / "data" / "Coffee Shop Sales.xlsx")
display(df["store_location"].value_counts())

df.head(3)

store_location
Hell's Kitchen     50735
Astoria            50599
Lower Manhattan    47782
Name: count, dtype: int64

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg


In [3]:
# Task 1: Calculate total revenue

df["line_revenue"] = (df["transaction_qty"] * df["unit_price"])
total_revenue_lower_manhantten = df[df["store_location"] == "Lower Manhattan"]["line_revenue"].sum()
total_revenue_hells_kitchen = df[df["store_location"] == "Hell's Kitchen"]["line_revenue"].sum()
total_revenue_astoria = df[df["store_location"] == "Astoria"]["line_revenue"].sum()

display(total_revenue_lower_manhantten)
display(total_revenue_hells_kitchen)
display(total_revenue_astoria)

total_combined_revenue = total_revenue_lower_manhantten+total_revenue_hells_kitchen+total_revenue_astoria
print(f"Total Revenue: ${total_combined_revenue:,.2f}")

np.float64(230057.25)

np.float64(236511.16999999998)

np.float64(232243.90999999997)

Total Revenue: $698,812.33


In [4]:
# Task 2: Total transactions

distinct_transactions = df["transaction_id"].nunique()
print(f"Distinct transactions: {distinct_transactions}")


Distinct transactions: 149116


In [5]:
# Task 3: Average transaction value

average_transaction_value = df["line_revenue"].sum()/distinct_transactions
print(f"Average transaction value: ${average_transaction_value}")

Average transaction value: $4.6863671906435265


In [6]:
lower_manhattan = df[
    df["store_location"] == "Lower Manhattan"
]

store_revenue = lower_manhattan["line_revenue"].sum()
store_transactions = lower_manhattan["transaction_id"].nunique()

store_atv = store_revenue / store_transactions

print(f"Lower Manhattan ATV: ${store_atv:.2f}")

lower_manhattan = df[
    df["store_location"] == "Astoria"
]

store_revenue = lower_manhattan["line_revenue"].sum()
store_transactions = lower_manhattan["transaction_id"].nunique()

store_atv = store_revenue / store_transactions

print(f"Astoria ATV: ${store_atv:.2f}")

lower_manhattan = df[
    df["store_location"] == "Hell's Kitchen"
]

store_revenue = lower_manhattan["line_revenue"].sum()
store_transactions = lower_manhattan["transaction_id"].nunique()

store_atv = store_revenue / store_transactions

print(f"Hell's Kitchen ATV: ${store_atv:.2f}")

Lower Manhattan ATV: $4.81
Astoria ATV: $4.59
Hell's Kitchen ATV: $4.66


### Which store has the highest total revenue?
Hell's Kitchen, with $236,511.17.

### Which store has the highest average transaction value?
Lower Manhattan, with $4.81 per recorded transaction ID.

### Does the store with the highest revenue also have the highest ATV?
No. Hell's Kitchen has the highest revenue, while Lower Manhattan has the highest average recorded transaction value.

### What might explain any differences?
Revenue depends on transaction volume and average transaction value. Hell's Kitchen records 50,735 transactions at an average of $4.66, while Lower Manhattan records 47,782 at an average of $4.81. Hell's Kitchen's greater recorded transaction volume offsets its lower average value. These figures describe the recorded transactions and do not establish the causes of customer behavior.


In [7]:
# Task 1: Find the top 10 products by revenue

products_by_revenue = df.groupby("product_detail")["line_revenue"].sum().sort_values(ascending=False)
products_by_revenue.head(10)


product_detail
Sustainably Grown Organic Lg    21151.75
Dark chocolate Lg               21006.00
Latte Rg                        19112.25
Cappuccino Lg                   17641.75
Morning Sunrise Chai Lg         17384.00
Latte                           17257.50
Jamaican Coffee River Lg        16481.25
Sustainably Grown Organic Rg    16233.75
Cappuccino                      15997.50
Brazilian Lg                    15109.50
Name: line_revenue, dtype: float64

In [8]:
# Task 2: Find the top 10 products by quantity sold

products_by_qty = df.groupby("product_detail")["transaction_qty"].sum().sort_values(ascending=False)
products_by_qty.head(10)


product_detail
Earl Grey Rg                   4708
Dark chocolate Lg              4668
Morning Sunrise Chai Rg        4643
Latte                          4602
Peppermint Rg                  4564
Columbian Medium Roast Rg      4547
Traditional Blend Chai Rg      4512
Latte Rg                       4497
Our Old Time Diner Blend Sm    4484
Serenity Green Tea Rg          4477
Name: transaction_qty, dtype: int64

In [9]:
# Task 3: Analyze product categories

revenue_by_category = df.groupby("product_category")["line_revenue"].sum().sort_values(ascending=False).to_frame(name="revenue")
revenue_by_category["percentage"] = (
    revenue_by_category["revenue"]
    / revenue_by_category["revenue"].sum()
    * 100
).round(2)
print(revenue_by_category)


                      revenue  percentage
product_category                         
Coffee              269952.45       38.63
Tea                 196405.95       28.11
Bakery               82315.64       11.78
Drinking Chocolate   72416.00       10.36
Coffee beans         40085.25        5.74
Branded              13607.00        1.95
Loose Tea            11213.60        1.60
Flavours              8408.80        1.20
Packaged Chocolate    4407.64        0.63


In [10]:
# Task 4: Compare product performance by store

# Step 1: Calculate total revenue for each product at each store
by_store = (
    df.groupby(["store_location", "product_detail"])["line_revenue"]
    .sum()
    .reset_index()
)

# Step 2: Sort by store (A-Z) and revenue (highest to lowest)
by_store = by_store.sort_values(
    ["store_location", "line_revenue"],
    ascending=[True, False]
)

# Step 3: Select the top 3 products within each store
by_store = (
    by_store.groupby("store_location")
    .head(3)
    .reset_index(drop=True)
)

# Step 4: Display results
display(by_store)

,store_location,product_detail,line_revenue
0,Astoria,Dark chocolate Lg,7897.50
1,Astoria,Sustainably Grown Organic Lg,7509.75
2,Astoria,Latte Rg,6358.00
3,Hell's Kitchen,Civet Cat,7380.00
4,Hell's Kitchen,Sustainably Grown Organic Lg,7329.25
5,Hell's Kitchen,Dark chocolate Lg,6534.00
6,Lower Manhattan,Dark chocolate Lg,6574.50
7,Lower Manhattan,Latte Rg,6387.75
8,Lower Manhattan,Sustainably Grown Organic Lg,6312.75


### Task 5: Write your business findings

Sustainably Grown Organic Lg generated the most individual-product revenue ($21,151.75).
Earl Grey Rg sold the most units (4,708).
Coffee generated $269,952.45, or 38.63% of total revenue.

In [11]:
# Task 1: Revenue by month
df["month"] = pd.to_datetime(df["transaction_date"]).dt.to_period('M')
by_month = df.groupby("month")["line_revenue"].sum()
by_month = by_month.sort_index()
print(by_month)

month
2023-01     81677.74
2023-02     76145.19
2023-03     98834.68
2023-04    118941.08
2023-05    156727.76
2023-06    166485.88
Freq: M, Name: line_revenue, dtype: float64


In [12]:
# Task 2: Revenue by day of the week
df["weekday"] = pd.to_datetime(df["transaction_date"]).dt.day_name()
days_per_weekday = df.groupby("weekday")["transaction_date"].nunique()
display(days_per_weekday)
by_weekday = df.groupby("weekday")["line_revenue"].sum()
by_weekday = by_weekday.sort_values(ascending=False)

by_weekday = by_weekday.to_frame(name = "total revenue").reset_index()
by_weekday["day count"] = by_weekday["weekday"].map(days_per_weekday)
by_weekday["revenue per day"] = by_weekday["total revenue"]/by_weekday["day count"]
print(by_weekday.sort_values(by="revenue per day", ascending=False))


weekday
Friday       26
Monday       26
Saturday     25
Sunday       26
Thursday     26
Tuesday      26
Wednesday    26
Name: transaction_date, dtype: int64

     weekday  total revenue  day count  revenue per day
0     Monday      101677.28         26      3910.664615
1     Friday      101373.00         26      3898.961538
6   Saturday       96894.48         25      3875.779200
2   Thursday      100767.78         26      3875.683846
3  Wednesday      100313.54         26      3858.213077
4    Tuesday       99455.94         26      3825.228462
5     Sunday       98330.31         26      3781.935000


In [13]:
# Task 3: Units sold by hour

df["time_of_day"] = df["transaction_time"].apply(lambda t: t.hour)
units_by_hour = df.groupby("time_of_day")["transaction_qty"].sum().sort_values(ascending=False)
print(units_by_hour)


time_of_day
10    26713
9     25370
8     25197
7     19449
11    14035
15    12923
14    12907
16    12881
17    12700
12    12690
13    12439
18    10826
19     8595
6      6865
20      880
Name: transaction_qty, dtype: int64


In [14]:
# Task 4: Revenue by hour
rev_by_hour = df.groupby("time_of_day")["line_revenue"].sum().sort_values(ascending=False)
print(rev_by_hour)

time_of_day
10    88673.39
9     85169.53
8     82699.87
7     63526.47
11    46319.14
15    41733.10
14    41304.74
16    41122.75
13    40367.45
12    40192.79
17    40134.31
18    34286.20
19    28446.68
6     21900.27
20     2935.64
Name: line_revenue, dtype: float64


In [15]:
# Task 5: Compare trends across locations

store_hour = (
    df.groupby(["store_location", "time_of_day"])["transaction_id"]
    .nunique()
    .reset_index(name="transactions")
    .sort_values(
        by=["store_location", "transactions"],
        ascending=[True, False]
    ) 
)


print(store_hour.groupby("store_location").head(5))

store_month = (
    df.groupby(["store_location", "month"])["line_revenue"].sum()
    .reset_index(name="total_revenue")
    .sort_values(
        by=["store_location", "total_revenue"],
        ascending=[True, False]
    ) 
)

print(store_month.groupby("store_location").head(5))


     store_location  time_of_day  transactions
3           Astoria           10          5291
2           Astoria            9          5083
1           Astoria            8          4966
0           Astoria            7          4181
9           Astoria           16          3599
17   Hell's Kitchen           10          6957
15   Hell's Kitchen            8          6909
16   Hell's Kitchen            9          6767
18   Hell's Kitchen           11          3598
14   Hell's Kitchen            7          3455
32  Lower Manhattan           10          6297
31  Lower Manhattan            9          5914
29  Lower Manhattan            7          5792
30  Lower Manhattan            8          5779
37  Lower Manhattan           15          3051
     store_location    month  total_revenue
5           Astoria  2023-06       55083.11
4           Astoria  2023-05       52428.76
3           Astoria  2023-04       39477.61
2           Astoria  2023-03       32835.43
0           Astoria  2023-01

## Task 6: Business Findings and Recommendations

### Key Findings

**1. All three stores reach their highest aggregate hourly transaction count at 10 AM.**

Across January–June 2023, the 10:00–10:59 AM bucket contains 6,957 transactions at Hell's Kitchen, 6,297 at Lower Manhattan, and 5,291 at Astoria. These counts are accumulated across six months.

**2. Store comparisons require attention to sales coverage.**

At 10 AM, Astoria records sales on 145 dates, compared with 181 dates for each of the other locations. Operating schedules and data completeness should be checked before interpreting these differences as evidence of stronger customer demand or greater staffing needs.

**3. Monthly revenue increases overall, with a February decline.**

Total revenue falls from $81,677.74 in January to $76,145.19 in February, a decrease of 6.77%. However, revenue per calendar day increases by 3.21%, showing why month length matters. Monthly total revenue then increases each month through June.

**4. Weekday rankings change after adjusting for observed days.**

Saturday ranks last in total revenue but third in revenue per day, averaging $3,875.78 across 25 Saturdays. Every other weekday has 26 observations. Saturday and Thursday are practically tied, differing by approximately $0.10 per day, so their ranking should not be treated as a meaningful performance gap.

### Business Recommendation

Management should review morning service capacity by location using confirmed operating schedules, existing staffing, labor costs, and customer wait times. Any staffing adjustment should begin as a limited trial with measurable service and cost outcomes.

Weekday sales comparisons should use revenue per observed day alongside total revenue. The small differences between weekday averages provide limited support for changing staffing or promotions based solely on weekday rankings.